<a href="https://colab.research.google.com/github/rekhaannapurna/Paddy-Disease-Detection/blob/main/Colab/MobileNetV2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rekhaannapurna","key":"21467604d1e5e4d024cd2f0716dd4824"}'}

In [2]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [3]:
!kaggle datasets list

ref                                                                 title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------------  -----------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
datascikhan/e-commerce-sales-and-customer-analytics                 🛒 E-Commerce Sales Analytics Dataset                29815424  2026-08-25 08:00:46.013000           2763         63                1  
dreaddevelopment/raptor-knee-widedense                              Knee MRI model weights: twelve findings            543320725  2026-08-23 04:12:20.703000            800         41           0.9375  
erfan4524/e-commerce-sales-data-analysis-and-eda                    E-Commerce Sales Data Analysis & EDA                 2209495  2026-08-28 15:12:17.040000           2217         43          

In [4]:
!kaggle datasets list -s paddy

ref                                                                title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
aman2000jaiswal/agriculture-crop-images                            Agriculture crop images                               62575817  2021-03-10 16:56:50.653000          15176        301  0.9411765        
abbas829/paddy-dataset                                             Paddy Dataset                                            21837  2026-03-24 14:10:30.400000            112         19  1                
ritikbompilwar/plantstressidentification                           Plant Stress Identification (Paddy Leaves)            19760543  2022-09-06 15:09:00.530000            554         16  0.7

In [5]:
!kaggle datasets download -d imbikramsaha/paddy-doctor

Dataset URL: https://www.kaggle.com/datasets/imbikramsaha/paddy-doctor
License(s): CC0-1.0
100% 1.02G/1.02G [00:12<00:00, 84.9MB/s]



In [6]:
!unzip -q paddy-doctor.zip -d /content/paddy-doctor

In [7]:
!find /content/paddy-doctor -maxdepth 2 -type d

/content/paddy-doctor
/content/paddy-doctor/paddy-disease-classification
/content/paddy-doctor/paddy-disease-classification/test_images
/content/paddy-doctor/paddy-disease-classification/train_images
/content/paddy-doctor/paddy-disease-classification/.ipynb_checkpoints


In [8]:
import glob
import os
import pandas as pd

try:
    import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *
from fastai.vision.all import *

set_seed(42)

# Our already-downloaded Paddy Doctor dataset
path = Path('/content/paddy-doctor/paddy-disease-classification')

# Train images
train_path = path / 'train_images'
train_files = get_image_files(train_path)

# Test images
test_path = path / 'test_images'
test_files = get_image_files(test_path).sorted()

# Check available files
print("Dataset path:", path)
print("Training images:", len(train_files))
print("Test images:", len(test_files))

# Train labels
train_df = pd.read_csv(path / 'train.csv')
print("Train CSV shape:", train_df.shape)

print("\nClass distribution:")
print(train_df.label.value_counts())

Dataset path: /content/paddy-doctor/paddy-disease-classification
Training images: 10407
Test images: 3469
Train CSV shape: (10407, 4)

Class distribution:
label
normal                      1764
blast                       1738
hispa                       1594
dead_heart                  1442
tungro                      1088
brown_spot                   965
downy_mildew                 620
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
Name: count, dtype: int64


In [9]:
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(0.2, seed=42),
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

dls = dblock.dataloaders(train_path)

In [10]:
dls = ImageDataLoaders.from_folder(
    train_path,
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

In [11]:
from fastai.vision.all import *
import torchvision.models as models
import torch.nn as nn

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Remove MobileNetV2's original classifier
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, dls.c)
)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 138MB/s]


In [12]:
learn = Learner(
    dls,
    model,
    loss_func=CrossEntropyLossFlat(),
    metrics=error_rate
).to_fp16()

In [13]:
learn.fine_tune(50)

epoch,train_loss,valid_loss,error_rate,time
0,0.985328,0.897398,0.272465,01:33


epoch,train_loss,valid_loss,error_rate,time
0,0.361245,0.288424,0.093224,01:32
1,0.241939,0.223203,0.069678,01:30
2,0.178391,0.186385,0.060067,01:44
3,0.144530,0.195291,0.059587,01:25
4,0.138427,0.195938,0.050937,01:18
5,0.123700,0.174541,0.049015,01:18
6,0.126823,0.260476,0.072081,01:18
7,0.145403,0.259615,0.069197,01:19
8,0.147231,0.215241,0.059587,01:19
9,0.142807,0.198919,0.056223,01:18


epoch,train_loss,valid_loss,error_rate,time
0,0.361245,0.288424,0.093224,01:32
1,0.241939,0.223203,0.069678,01:30
2,0.178391,0.186385,0.060067,01:44
3,0.144530,0.195291,0.059587,01:25
4,0.138427,0.195938,0.050937,01:18
5,0.123700,0.174541,0.049015,01:18
6,0.126823,0.260476,0.072081,01:18
7,0.145403,0.259615,0.069197,01:19
8,0.147231,0.215241,0.059587,01:19
9,0.142807,0.198919,0.056223,01:18


In [14]:
learn.validate()

[0.14286594092845917, 0.02498798631131649]

In [16]:
learn_loss , learn_error = learn.validate()

print("Accuracy:", 1-learn_error)

Accuracy: 0.9750120136886835


In [15]:
# Get predictions on validation set
probs, target = learn.get_preds(dl=dls.valid)
error_rate(probs, target)

TensorBase(0.0250)

In [17]:
# Get TTA predictions on validation set
probs, target = learn.tta(dl=dls.valid)
error_rate(probs, target)

epoch,train_loss,valid_loss,error_rate,time


<div></div>

TensorBase(0.0235)

In [18]:
# test images
test_path = path/'test_images'
test_files = get_image_files(test_path).sorted()
test_classes = [f.parent.name for f in test_files]
probs, _ = learn.tta(dl=dls.test_dl(test_files))
preds = probs.argmax(dim=1)
pred_classes = dls.vocab[preds]

epoch,train_loss,valid_loss,error_rate,time


<div></div>

In [19]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

cls_report = classification_report(test_classes, pred_classes,
                                   digits=5)
print(cls_report)
acc = accuracy_score(test_classes, pred_classes)

                          precision    recall  f1-score   support

   bacterial_leaf_blight    0.00000   0.00000   0.00000       0.0
   bacterial_leaf_streak    0.00000   0.00000   0.00000       0.0
bacterial_panicle_blight    0.00000   0.00000   0.00000       0.0
                   blast    0.00000   0.00000   0.00000       0.0
              brown_spot    0.00000   0.00000   0.00000       0.0
              dead_heart    0.00000   0.00000   0.00000       0.0
            downy_mildew    0.00000   0.00000   0.00000       0.0
                   hispa    0.00000   0.00000   0.00000       0.0
                  normal    0.00000   0.00000   0.00000       0.0
             test_images    0.00000   0.00000   0.00000    3469.0
                  tungro    0.00000   0.00000   0.00000       0.0

                accuracy                        0.00000    3469.0
               macro avg    0.00000   0.00000   0.00000    3469.0
            weighted avg    0.00000   0.00000   0.00000    3469.0



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

In [20]:
temp = pd.DataFrame({"y_true":test_classes,
                      "y_pred":pred_classes})
temp.to_csv('result.csv', index=False)
temp

,y_true,y_pred
0,test_images,hispa
1,test_images,normal
2,test_images,blast
3,test_images,blast
4,test_images,blast
...,...,...
3464,test_images,dead_heart
3465,test_images,hispa
3466,test_images,normal
3467,test_images,bacterial_leaf_streak


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
from pathlib import Path

backup_path = Path('/content/drive/MyDrive/Paddy_Disease_Project')
backup_path.mkdir(parents=True, exist_ok=True)

print("Backup folder:", backup_path)

Backup folder: /content/drive/MyDrive/Paddy_Disease_Project


In [23]:
learn.export(
    backup_path / 'MobileNetV2_paddy_baseline.pkl'
)

print("Model exported successfully!")

Model exported successfully!


In [26]:
learn.save(
    str(backup_path / 'MobileNetV2_paddy_baseline')
)

print("Model weights saved successfully!")

Model weights saved successfully!


In [27]:
!ls -lh /content/drive/MyDrive/Paddy_Disease_Project

total 62M
-rw------- 1 root root 9.4M Sep  5 10:28 MobileNetV2_paddy_baseline.pkl
-rw------- 1 root root  26M Sep  5 10:29 MobileNetV2_paddy_baseline.pth
-rw------- 1 root root  26M Sep  5 10:28 resnet34_paddy_baseline.pth
